# Test/train split

This notebook splits the data (expected to be in `../studres/data/*.csv`) and writes the outputs to `../dataset-split/`; the splits are further broken apart in ~300MB sized files due to github limits.

**NOTE:** this notebook contains one data cleaning step of dropping rides with pickups outside of Manhattan

In [1]:
import datetime
from glob import glob

import pandas as pd
import sklearn as sk

In [2]:
def dbg(x):
    print(x)
    return x

rides = pd.concat([
    pd.read_csv(
        dbg(f),
        dtype=dict(
            VendorID=int,
            passenger_count=float,
            trip_distance=float,
            RatecodeID="Int64", # some are missing
            store_and_fwd_flag=str,
            PULocationID=int,
            DOLocationID=int,
            payment_type=int,
            fare_amount=float,
            extra=float,
            mta_tax=float,
            tip_amount=float,
            tolls_amount=float,
            improvement_surcharge=float,
            total_amount=float,
            congestion_surcharge=float,
            Airport_fee=float,
        ),
        parse_dates=[
            "tpep_pickup_datetime",
            "tpep_dropoff_datetime",
        ],
    )
    for f in sorted(glob("../studres/data/*.csv"))
], ignore_index=True)

../data/nyc_taxi_2024-01.csv
../data/nyc_taxi_2024-02.csv
../data/nyc_taxi_2024-03.csv
../data/nyc_taxi_2024-04.csv
../data/nyc_taxi_2024-05.csv
../data/nyc_taxi_2024-06.csv
../data/nyc_taxi_2024-07.csv
../data/nyc_taxi_2024-08.csv
../data/nyc_taxi_2024-09.csv
../data/nyc_taxi_2024-10.csv
../data/nyc_taxi_2024-11.csv
../data/nyc_taxi_2024-12.csv



Drop rides that start outside of Manhattan

In [3]:
zones = pd.read_csv("../dataset-split/taxi_zone_lookup.csv")
zones = set(zones[zones["Borough"] == "Manhattan"]["LocationID"])

rides = rides[rides["PULocationID"].isin(zones)]


# Do the split

* Group by week of the month
  * This grouping strikes a balance of granularity and not being correlated
    to the target variables; compared to day of the week, hour of day, which
    is expected to be correlated.
* Find groups such that ride duration has similar distributions between
  test and train datasets.
  * Using the target variable is ideal, but since we have two targets,
    demand and revenue, trip duration is reasonably correlated to both
    of these:
    * longer, in time, trips probably reflect traffic and thus demand.
    * longer, in time, trips will cost more.
  * Another reasonable choice is distance, but that's likely less correlated
    to demand.

In [4]:
[(train_idx, test_idx)] = sk.model_selection.GroupShuffleSplit(
    n_splits=1,
    test_size=0.2,
    random_state=2718,
).split(
    rides,
    y=(
        rides["tpep_dropoff_datetime"] - rides["tpep_pickup_datetime"]
    ).map(lambda x: x.seconds / 60).astype(int),
    groups=(
        rides["tpep_pickup_datetime"].map(lambda x: x.day) / 7
    ).astype(int),
)

In [ ]:
# https://stackoverflow.com/a/17511341
def ceildiv(a, b):
    return -(a // -b)

def save_split(df, prefix, rows_per_file=2300000):
    for i in range(0, ceildiv(len(df), rows_per_file)):
        df[i*rows_per_file:(i+1)*rows_per_file].to_csv(dbg(f"{prefix}_{i:02}.csv"), index=False)

# a sample dataset for faster testing
save_split(rides.iloc[train_idx].sample(1800000), "../dataset-split/sample")
# real training dataset
save_split(rides.iloc[train_idx], "../dataset-split/train")
# test dataset, only used at the very end
save_split(rides.iloc[test_idx], "../dataset-split/test")